In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Kaggle notebook code (GPU recommended)
# ------------------------------------

import json, os, random 
from collections import Counter, defaultdict

import torch
from torch.utils.data import DataLoader

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    # Trainer,
    DataCollatorWithPadding,
    set_seed
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

TRAIN_PATH = "/kaggle/input/data4good/train.json"  # provided path in this environment; on Kaggle set accordingly

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data4good/train.json
/kaggle/input/data4good-test/test.json
/kaggle/input/data4good-trainset-gpt/trainset-gpt5.1.csv


In [ ]:
# -------------------------
# 1) Load + Stratified Split
# -------------------------
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# expected fields in each row: answer, type, context, question
# type in {"factual","contradiction","irrelevant"}
labels = [ex["type"] for ex in data]
print("N =", len(data))
print("Label distribution:", Counter(labels))

idx = np.arange(len(data))

# 80 / 10 / 10 stratified
idx_train, idx_tmp = train_test_split(
    idx, test_size=0.20, random_state=SEED, stratify=labels
)
tmp_labels = [labels[i] for i in idx_tmp]
idx_val, idx_test = train_test_split(
    idx_tmp, test_size=0.50, random_state=SEED, stratify=tmp_labels
)

def subset(idxs):
    return [data[i] for i in idxs]

train_data = subset(idx_train)
val_data   = subset(idx_val)
test_data  = subset(idx_test)

print("\nSplit sizes:", {"train": len(train_data), "val": len(val_data), "test": len(test_data)})
print("Train dist:", Counter([x["type"] for x in train_data]))
print("Val dist:",   Counter([x["type"] for x in val_data]))
print("Test dist:",  Counter([x["type"] for x in test_data]))

In [ ]:

import os, json
from pathlib import Path
from collections import Counter

# Choose an output dir (Kaggle-friendly)
OUT_DIR = Path("./splits_80_10_10")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_json(path: Path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# Save
save_json(OUT_DIR / "train_80.json", train_data)
save_json(OUT_DIR / "val_10.json",   val_data)
save_json(OUT_DIR / "test_10.json",  test_data)

# Optional: also save as JSONL (often easier to stream)
def save_jsonl(path: Path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(OUT_DIR / "train_80.jsonl", train_data)
save_jsonl(OUT_DIR / "val_10.jsonl",   val_data)
save_jsonl(OUT_DIR / "test_10.jsonl",  test_data)

print("Saved to:", OUT_DIR.resolve())
for name, rows in [("train", train_data), ("val", val_data), ("test", test_data)]:
    print(f"{name:5s} n={len(rows):5d} dist={Counter([x['type'] for x in rows])}")



In [ ]:



# If you want to save splits:
# with open("train_80.json","w",encoding="utf-8") as f: json.dump(train_data,f,ensure_ascii=False)
# with open("val_10.json","w",encoding="utf-8") as f: json.dump(val_data,f,ensure_ascii=False)
# with open("test_10.json","w",encoding="utf-8") as f: json.dump(test_data,f,ensure_ascii=False)

# -------------------------
# Helpers: build MNLI pairs + label mapping
# -------------------------
CLASS_NAMES = ["factual", "contradiction", "irrelevant"]

def build_premise(ex):
    # You can tweak this, but keep it consistent across runs
    ctx = (ex.get("context") or "").strip()
    q   = (ex.get("question") or "").strip()
    if ctx:
        return f"{ctx}\n\nQuestion: {q}"
    return f"Question: {q}"

def build_hypothesis(ex):
    return (ex.get("answer") or "").strip()

def get_mnli_label_ids(model_config):
    """
    Returns dict: {"entailment": id, "contradiction": id, "neutral": id}
    Works across models where config.label2id may be inconsistent/cased.
    """
    l2i = model_config.label2id or {}
    # normalize
    norm = {str(k).lower(): int(v) for k, v in l2i.items()}
    out = {}

    # common names in MNLI configs
    for key in ["entailment", "contradiction", "neutral"]:
        if key in norm:
            out[key] = norm[key]

    # sometimes keys are like "LABEL_0"/"LABEL_1"/"LABEL_2"
    # and config.id2label provides meaning
    if len(out) < 3 and getattr(model_config, "id2label", None):
        i2l = {int(k): str(v).lower() for k, v in model_config.id2label.items()}
        for i, name in i2l.items():
            if "entail" in name:
                out["entailment"] = i
            elif "contra" in name:
                out["contradiction"] = i
            elif "neutral" in name:
                out["neutral"] = i

    # final fallback (MNLI default often: 0=contradiction,1=neutral,2=entailment)
    if len(out) < 3:
        out = {"contradiction": 0, "neutral": 1, "entailment": 2}

    return out

# dataset label mapping:
# factual -> entailment, contradiction -> contradiction, irrelevant -> neutral
def to_mnli_target_id(ex, mnli_ids):
    t = ex["type"]
    if t == "factual":
        return mnli_ids["entailment"]
    if t == "contradiction":
        return mnli_ids["contradiction"]
    if t == "irrelevant":
        return mnli_ids["neutral"]
    raise ValueError(f"Unknown type: {t}")

def preds_to_task_label(pred_mnli_id, mnli_ids):
    # reverse mapping MNLI decision -> task class
    if pred_mnli_id == mnli_ids["entailment"]:
        return "factual"
    if pred_mnli_id == mnli_ids["contradiction"]:
        return "contradiction"
    if pred_mnli_id == mnli_ids["neutral"]:
        return "irrelevant"
    # if model outputs something unexpected, call it irrelevant
    return "irrelevant"

def compute_metrics_task(y_true, y_pred, title=""):
    # overall
    overall_acc = float(np.mean([a == b for a, b in zip(y_true, y_pred)]))
    overall_f1_macro = float(f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro"))

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in CLASS_NAMES:
        idxs = [i for i, yt in enumerate(y_true) if yt == c]
        if len(idxs) == 0:
            per_class_acc[c] = None
        else:
            per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs]))

    report = classification_report(
        y_true, y_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0
    )

    # print clean summary
    print("\n" + "="*80)
    if title:
        print(title)
    print(f"Overall Accuracy: {overall_acc:.4f}")
    print(f"Overall F1 (macro): {overall_f1_macro:.4f}")
    print("\nPer-class Accuracy:")
    for c in CLASS_NAMES:
        v = per_class_acc[c]
        print(f"  {c:14s} {('NA' if v is None else f'{v:.4f}')}")
    print("\nPer-class F1:")
    for c in CLASS_NAMES:
        print(f"  {c:14s} {report[c]['f1-score']:.4f}")

    return {
        "overall_accuracy": overall_acc,
        "overall_f1_macro": overall_f1_macro,
        "per_class_accuracy": per_class_acc,
        "per_class_f1": {c: float(report[c]["f1-score"]) for c in CLASS_NAMES},
    }

# -------------------------
# 2) Evaluate DeBERTa-MNLI + 3 other MNLI models (NO finetuning)
# -------------------------
MNLI_MODELS = [
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli" , # strong MNLI-style NLI model
    "microsoft/deberta-large-mnli",   # DeBERTa MNLI
    "roberta-large-mnli",             # RoBERTa MNLI
    "facebook/bart-large-mnli"      # BART MNLI
]

@torch.no_grad()
def eval_mnli_model(model_name, eval_examples, batch_size=16, max_length=256):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nLoading: {model_name} on {device}")

    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()

    mnli_ids = get_mnli_label_ids(model.config)

    # build tokenized batches
    premises = [build_premise(ex) for ex in eval_examples]
    hypos    = [build_hypothesis(ex) for ex in eval_examples]
    y_true   = [ex["type"] for ex in eval_examples]

    # simple dataloader over indices
    indices = np.arange(len(eval_examples))
    loader = DataLoader(indices, batch_size=batch_size, shuffle=False)

    y_pred = []
    for batch_idx in loader:
        batch_idx = batch_idx.numpy().tolist()
        p = [premises[i] for i in batch_idx]
        h = [hypos[i] for i in batch_idx]
        enc = tok(p, h, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits
        pred_ids = torch.argmax(logits, dim=-1).detach().cpu().numpy().tolist()
        y_pred.extend([preds_to_task_label(pid, mnli_ids) for pid in pred_ids])

    return y_true, y_pred
import pandas as pd
from pathlib import Path

OUT_DIR = Path("./baseline_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def metrics_to_row(model_name, split_name, metrics):
    return {
        "model": model_name,
        "split": split_name,
        "overall_accuracy": metrics["overall_accuracy"],
        "overall_f1_macro": metrics["overall_f1_macro"],
        "acc_factual": metrics["per_class_accuracy"]["factual"],
        "acc_contradiction": metrics["per_class_accuracy"]["contradiction"],
        "acc_irrelevant": metrics["per_class_accuracy"]["irrelevant"],
        "f1_factual": metrics["per_class_f1"]["factual"],
        "f1_contradiction": metrics["per_class_f1"]["contradiction"],
        "f1_irrelevant": metrics["per_class_f1"]["irrelevant"],
    }

def run_baselines_and_save_csv():
    rows = []

    for m in MNLI_MODELS:
        # ---- Validation ----
        y_true_v, y_pred_v = eval_mnli_model(m, val_data, batch_size=16)
        metrics_v = compute_metrics_task(
            y_true_v, y_pred_v,
            title=f"[NO FT] {m} on VAL (10%)"
        )
        rows.append(metrics_to_row(m, "val", metrics_v))

        # ---- Test ----
        y_true_t, y_pred_t = eval_mnli_model(m, test_data, batch_size=16)
        metrics_t = compute_metrics_task(
            y_true_t, y_pred_t,
            title=f"[NO FT] {m} on TEST (10%)"
        )
        rows.append(metrics_to_row(m, "test", metrics_t))

    df = pd.DataFrame(rows)

    # Save combined CSV
    df.to_csv(OUT_DIR / "mnli_baselines_all.csv", index=False)

    # Also save split-wise CSVs (useful for tables in paper)
    df[df["split"] == "val"].to_csv(
        OUT_DIR / "mnli_baselines_val.csv", index=False
    )
    df[df["split"] == "test"].to_csv(
        OUT_DIR / "mnli_baselines_test.csv", index=False
    )

    print("Saved CSVs to:", OUT_DIR.resolve())
    return df

# run
baseline_df = run_baselines_and_save_csv()
baseline_df


## Finetuning

In [ ]:
import os, json, gc, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
gc.collect(); torch.cuda.empty_cache()

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # use public model for reproducibility
# MODEL_NAME = "microsoft/deberta-v3-base-mnli"  # if you want faster + less OOM risk

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

MAX_LENGTH = 192
LR = 2e-5
EPOCHS = 2
TRAIN_BS = 2
EVAL_BS = 8
GRAD_ACC = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

SPLIT_DIR = "./splits_80_10_10"     # where you saved train_80.json etc.
OUT_DIR = "./outputs_80_10_10"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# Load split json files
# -----------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json(os.path.join(SPLIT_DIR, "train_80.json"))
val_data   = load_json(os.path.join(SPLIT_DIR, "val_10.json"))
test_data  = load_json(os.path.join(SPLIT_DIR, "test_10.json"))

def to_df(records, with_labels=True):
    df = pd.DataFrame(records)

    ctx = df.get("context", pd.Series([""]*len(df))).fillna("").astype(str)
    q   = df.get("question", pd.Series([""]*len(df))).fillna("").astype(str)
    ans = df["answer"].fillna("").astype(str)

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = ans

    if with_labels:
        df["label"] = df["type"].astype(str).str.lower().map(label2id).astype(int)

    return df

train_df = to_df(train_data, with_labels=True)
val_df   = to_df(val_data, with_labels=True)
test_df  = to_df(test_data, with_labels=True)

print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist:", train_df["type"].value_counts().to_dict())
print("Val dist:", val_df["type"].value_counts().to_dict())
print("Test dist:", test_df["type"].value_counts().to_dict())

# -----------------------
# Tokenize
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_dataset(df, with_labels=True):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    keep = ["premise_text", "hypothesis_text"] + (["label"] if with_labels else [])
    remove_cols = [c for c in ds.column_names if c not in keep]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)

    if with_labels:
        ds = ds.rename_column("label", "labels")
        cols = ["input_ids", "attention_mask", "labels"]
    else:
        cols = ["input_ids", "attention_mask"]

    ds.set_format(type="torch", columns=cols)
    return ds

train_ds = tokenize_dataset(train_df, with_labels=True)
val_ds   = tokenize_dataset(val_df, with_labels=True)
test_ds  = tokenize_dataset(test_df, with_labels=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Model
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

# memory saver
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()
model.config.use_cache = False

# -----------------------
# Metrics (paper required)
# -----------------------
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = metrics_from_preds(labels.tolist(), preds.tolist())
    # Trainer expects flat numeric values
    return {"accuracy": m["overall_accuracy"], "f1_macro": m["overall_f1_macro"]}

# -----------------------
# TrainingArguments (handles eval_strategy API mismatch)
# -----------------------
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

args_kwargs = dict(
    output_dir=os.path.join(OUT_DIR, "ft_model"),
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)
args_kwargs[eval_key] = "epoch"
args_kwargs = {k:v for k,v in args_kwargs.items() if k in allowed}

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_trainer,
)

gc.collect(); torch.cuda.empty_cache()
trainer.train()

# -----------------------
# Evaluate on VAL + TEST, save metrics + predictions
# -----------------------
def predict_and_save(split_name, df, ds):
    pred = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    conf = probs.max(axis=-1)
    true_ids = df["label"].values

    # metrics
    met = metrics_from_preds(true_ids.tolist(), pred_ids.tolist())
    met_row = {"model": MODEL_NAME, "split": split_name, **met}

    # predictions csv
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = [id2label[i] for i in pred_ids]
    out["pred_conf"] = conf
    for i,lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_{split_name}.csv")
    out.to_csv(pred_path, index=False)

    return met_row, pred_path

val_metrics_row, val_pred_path = predict_and_save("val", val_df, val_ds)
test_metrics_row, test_pred_path = predict_and_save("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_metrics_row, test_metrics_row])
metrics_path = os.path.join(OUT_DIR, "finetune_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df


In [ ]:
import os, gc, torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

gc.collect()
torch.cuda.empty_cache()

# Path where Trainer saved your best checkpoint
SAVED_DIR = os.path.join('/kaggle/working/outputs_80_10_10', "ft_model/checkpoint-1051")  # same as output_dir used in training

# Load tokenizer + fine-tuned model from disk
tokenizer = AutoTokenizer.from_pretrained(SAVED_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(SAVED_DIR)

# If you want to enforce label names (optional, but helps readability)
model.config.id2label = id2label
model.config.label2id = label2id

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Minimal args just for prediction (eval batch size matters)
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

pred_args_kwargs = dict(
    output_dir=os.path.join('/kaggle/working/outputs_80_10_10', "pred_only"),
    per_device_eval_batch_size=EVAL_BS,
    fp16=True,
    report_to="none",
)
pred_args_kwargs[eval_key] = "no"
pred_args_kwargs = {k:v for k,v in pred_args_kwargs.items() if k in allowed}
pred_args = TrainingArguments(**pred_args_kwargs)

pred_trainer = Trainer(
    model=model,
    args=pred_args,
    data_collator=data_collator,
)

LABELS = ["factual", "contradiction", "irrelevant"]

def metrics_table(y_true_str, y_pred_str):
    # overall
    overall_acc = float(np.mean(np.array(y_true_str) == np.array(y_pred_str)))
    overall_f1  = float(f1_score(y_true_str, y_pred_str, labels=LABELS, average="macro", zero_division=0))

    # per-class accuracy
    per_class_acc = {}
    for c in LABELS:
        idxs = np.where(np.array(y_true_str) == c)[0]
        per_class_acc[c] = float(np.mean(np.array(y_pred_str)[idxs] == c)) if len(idxs) else None

    # per-class f1
    rep = classification_report(
        y_true_str, y_pred_str, labels=LABELS, output_dict=True, zero_division=0
    )

    row = {
        "overall_accuracy": overall_acc,
        "overall_f1_macro": overall_f1,
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(rep["factual"]["f1-score"]),
        "f1_contradiction": float(rep["contradiction"]["f1-score"]),
        "f1_irrelevant": float(rep["irrelevant"]["f1-score"]),
    }
    return row

@torch.no_grad()
def predict_split(split_name, df, ds):
    pred = pred_trainer.predict(ds)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    pred_ids = probs.argmax(axis=-1)
    pred_labels = [id2label[int(i)] for i in pred_ids]

    true_labels = df["type"].astype(str).str.lower().tolist()

    # metrics
    met = metrics_table(true_labels, pred_labels)
    met_row = {"model": SAVED_DIR, "split": split_name, **met}

    # save predictions
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = pred_labels
    out["pred_conf"] = probs.max(axis=-1)
    for i, lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_{split_name}_LOADED.csv")
    out.to_csv(pred_path, index=False)

    print(split_name, met)
    return met_row, pred_path

# Predict + metrics on VAL and TEST using LOADED model
val_row, val_pred_path = predict_split("val", val_df, val_ds)
test_row, test_pred_path = predict_split("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_row, test_row])
metrics_path = os.path.join('/kaggle/working/outputs_80_10_10', "finetune_metrics_LOADED.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df


In [ ]:
import os, json, gc, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
gc.collect(); torch.cuda.empty_cache()

MODEL_NAME = "facebook/bart-large-mnli"

LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

MAX_LENGTH = 150
LR = 2e-5
EPOCHS = 2
TRAIN_BS = 2
EVAL_BS = 8
GRAD_ACC = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

SPLIT_DIR = "./splits_80_10_10"     # where you saved train_80.json etc.
OUT_DIR = "./outputs_80_10_10"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------
# Load split json files
# -----------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_json(os.path.join(SPLIT_DIR, "train_80.json"))
val_data   = load_json(os.path.join(SPLIT_DIR, "val_10.json"))
test_data  = load_json(os.path.join(SPLIT_DIR, "test_10.json"))

def to_df(records, with_labels=True):
    df = pd.DataFrame(records)

    ctx = df.get("context", pd.Series([""]*len(df))).fillna("").astype(str)
    q   = df.get("question", pd.Series([""]*len(df))).fillna("").astype(str)
    ans = df["answer"].fillna("").astype(str)

    df["premise_text"] = (ctx + "\n\nQuestion: " + q).str.strip()
    df["hypothesis_text"] = ans

    if with_labels:
        df["label"] = df["type"].astype(str).str.lower().map(label2id).astype(int)

    return df

train_df = to_df(train_data, with_labels=True)
val_df   = to_df(val_data, with_labels=True)
test_df  = to_df(test_data, with_labels=True)

print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train dist:", train_df["type"].value_counts().to_dict())
print("Val dist:", val_df["type"].value_counts().to_dict())
print("Test dist:", test_df["type"].value_counts().to_dict())

# -----------------------
# Tokenize
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_dataset(df, with_labels=True):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    def _tok(batch):
        return tokenizer(
            batch["premise_text"],
            batch["hypothesis_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    keep = ["premise_text", "hypothesis_text"] + (["label"] if with_labels else [])
    remove_cols = [c for c in ds.column_names if c not in keep]
    ds = ds.map(_tok, batched=True, remove_columns=remove_cols)

    if with_labels:
        ds = ds.rename_column("label", "labels")
        cols = ["input_ids", "attention_mask", "labels"]
    else:
        cols = ["input_ids", "attention_mask"]

    ds.set_format(type="torch", columns=cols)
    return ds

train_ds = tokenize_dataset(train_df, with_labels=True)
val_ds   = tokenize_dataset(val_df, with_labels=True)
test_ds  = tokenize_dataset(test_df, with_labels=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Model
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

# memory saver
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()
model.config.use_cache = False

# -----------------------
# Metrics (paper required)
# -----------------------
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

from transformers import EvalPrediction

def compute_metrics_trainer(eval_pred: EvalPrediction):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    # BART can return a tuple; first element is logits
    if isinstance(preds, (tuple, list)):
        preds = preds[0]

    pred_ids = np.argmax(preds, axis=-1)
    m = metrics_from_preds(labels.tolist(), pred_ids.tolist())
    return {"accuracy": m["overall_accuracy"], "f1_macro": m["overall_f1_macro"]}

# -----------------------
# TrainingArguments (handles eval_strategy API mismatch)
# -----------------------
import inspect
sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())
eval_key = "eval_strategy" if "eval_strategy" in allowed else "evaluation_strategy"

args_kwargs = dict(
    output_dir=os.path.join(OUT_DIR, "ft_model_bart"),
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)
args_kwargs[eval_key] = "epoch"
args_kwargs = {k:v for k,v in args_kwargs.items() if k in allowed}

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_trainer,
)

gc.collect(); torch.cuda.empty_cache()
trainer.train()

# -----------------------
# Evaluate on VAL + TEST, save metrics + predictions
# -----------------------
def predict_and_save(split_name, df, ds):
    pred = trainer.predict(ds)
    raw = pred.predictions
    if isinstance(raw, (tuple, list)):
        raw = raw[0]
    probs = torch.softmax(torch.from_numpy(raw).float(), dim=-1).numpy()

    pred_ids = probs.argmax(axis=-1)
    conf = probs.max(axis=-1)
    true_ids = df["label"].values

    # metrics
    met = metrics_from_preds(true_ids.tolist(), pred_ids.tolist())
    met_row = {"model": MODEL_NAME, "split": split_name, **met}

    # predictions csv
    out = df.copy()
    out["pred_id"] = pred_ids
    out["pred_label"] = [id2label[i] for i in pred_ids]
    out["pred_conf"] = conf
    for i,lbl in enumerate(LABELS):
        out[f"prob_{lbl}"] = probs[:, i]
    pred_path = os.path.join(OUT_DIR, f"predictions_bart_{split_name}.csv")
    out.to_csv(pred_path, index=False)

    return met_row, pred_path

val_metrics_row, val_pred_path = predict_and_save("val", val_df, val_ds)
test_metrics_row, test_pred_path = predict_and_save("test", test_df, test_ds)

metrics_df = pd.DataFrame([val_metrics_row, test_metrics_row])
metrics_path = os.path.join(OUT_DIR, "finetune_metrics_bart.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Saved:")
print(" -", metrics_path)
print(" -", val_pred_path)
print(" -", test_pred_path)
metrics_df

## Context regime split

In [34]:
test_df = pd.read_csv("/kaggle/working/outputs_80_10_10/predictions_test_LOADED.csv")
val_df = pd.read_csv("/kaggle/working/outputs_80_10_10/predictions_val_LOADED.csv")

In [22]:
from sklearn.metrics import accuracy_score, f1_score
LABELS = ["factual", "contradiction", "irrelevant"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}
def metrics_from_preds(y_true_ids, y_pred_ids):
    y_true = [id2label[i] for i in y_true_ids]
    y_pred = [id2label[i] for i in y_pred_ids]

    overall_acc = accuracy_score(y_true, y_pred)
    per_class = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)

    # per-class accuracy: correct within that class / total in that class
    per_class_acc = {}
    for c in LABELS:
        idxs = [i for i,t in enumerate(y_true) if t == c]
        per_class_acc[c] = float(np.mean([y_pred[i] == c for i in idxs])) if idxs else None

    out = {
        "overall_accuracy": float(overall_acc),
        "overall_f1_macro": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "acc_factual": per_class_acc["factual"],
        "acc_contradiction": per_class_acc["contradiction"],
        "acc_irrelevant": per_class_acc["irrelevant"],
        "f1_factual": float(per_class["factual"]["f1-score"]),
        "f1_contradiction": float(per_class["contradiction"]["f1-score"]),
        "f1_irrelevant": float(per_class["irrelevant"]["f1-score"]),
    }
    return out

In [31]:
def metrics_by_context_na(df):
    results = {}

    # context IS NA
    df_na = df[df["context"].isna()]
    results["context_is_na"] = metrics_from_preds(
        df_na.label.tolist(),
        df_na.pred_id.tolist()
    )
    print(df_na.shape)
    # context IS NOT NA
    df_not_na = df[df["context"].notna()]
    results["context_not_na"] = metrics_from_preds(
        df_not_na.label.tolist(),
        df_not_na.pred_id.tolist()
    )
    print(df_not_na.shape)

    return results


In [32]:
res = metrics_by_context_na(test_df)

pd.concat(
    {
        k: pd.DataFrame([v])
        for k, v in res.items()
    }
)


(183, 13)
(1920, 13)


,,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
context_is_na,0,0.994536,0.988140,1.000000,0.937500,1.00000,0.996678,0.967742,1.000000
context_not_na,0,0.992188,0.983043,0.997491,0.963855,0.96875,0.995305,0.969697,0.984127


In [35]:
res = metrics_by_context_na(val_df)

pd.concat(
    {
        k: pd.DataFrame([v])
        for k, v in res.items()
    }
)


(181, 13)
(1921, 13)


,,overall_accuracy,overall_f1_macro,acc_factual,acc_contradiction,acc_irrelevant,f1_factual,f1_contradiction,f1_irrelevant
context_is_na,0,0.966851,0.924451,1.000000,0.714286,1.000000,0.983498,0.833333,0.956522
context_not_na,0,0.993233,0.985280,0.995609,0.968944,0.993976,0.995921,0.965944,0.993976
